# 04 — All-layer learned-mixture comparison
Use all 12 frozen transformer layers and learn four softmax mixtures (48 scalar logits) for the unchanged common decoder. Compare the result with the fixed `[2,5,8,11]` runs from notebook 03.

In [ ]:
# Fresh-kernel bootstrap. Edit only these settings; cached Drive assets are reused.
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
ACCEPT_COD10K_NONCOMMERCIAL_LICENSE = True

from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch
project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[dev,notebooks]'], check=True)
bootstrap_env = os.environ.copy()
dino_weights = Path('/content/drive/MyDrive/cod-ssl/checkpoints/dinov3_vitb16.pth')
if not dino_weights.is_file():
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
command = [sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
           '--project-dir', str(project_dir), '--state-file', str(state_file)]
if not ACCEPT_COD10K_NONCOMMERCIAL_LICENSE:
    raise PermissionError('Review https://github.com/DengPingFan/SINet#9-license before accepting.')
command += ['--ensure-training-data', '--accept-noncommercial-license']
subprocess.run(command, cwd=project_dir, env=bootstrap_env, check=True)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = Path(state['project_dir'])
DATA_ROOT = Path(state['data_root'])
RUNS_ROOT = Path(state['runs_root'])
COMPARISONS_ROOT = Path(state['comparisons_root'])
TRAIN_MANIFEST = Path(state['train_manifest'])
os.chdir(PROJECT_DIR)
print('Ready on', state['gpu'])

In [ ]:
# Validate the same decontaminated protocol used by notebook 03.
exclusions = str(PROJECT_DIR / 'configs/dataset_exclusions.csv')
receipt = DATA_ROOT / 'validation/dataset_validation_v3.json'
commands = [
    [sys.executable, 'scripts/bootstrap_test_data.py', '--data-root', str(DATA_ROOT),
     '--manifest-dir', 'manifests', '--exclusions', exclusions, '--accept-noncommercial-license'],
    [sys.executable, 'scripts/validate_dataset.py', '--manifest-dir', 'manifests',
     '--exclusions', exclusions, '--receipt', str(receipt)],
]
for command in commands: subprocess.run(command, cwd=PROJECT_DIR, check=True)
os.sync()

In [ ]:
# Stable Drive paths and resumable 40-epoch training.
TAG = 'phase1_seed42_all12_mix4'
DINO_RUN = RUNS_ROOT / f'{TAG}_dinov3_vitb16'
VJEPA_RUN = RUNS_ROOT / f'{TAG}_vjepa21_vitb16'
MIXTURE_COMPARISON = COMPARISONS_ROOT / f'{TAG}_comparison'
CONFIGS = [
    ('configs/all_layers_dinov3_vitb16.yaml', DINO_RUN),
    ('configs/all_layers_vjepa21_vitb16.yaml', VJEPA_RUN),
]
def train_or_resume(config, run_dir):
    last = run_dir / 'checkpoints/last.pt'
    if last.is_file(): print('Completed, skipping:', run_dir.name); return
    checkpoints = sorted((run_dir / 'checkpoints').glob('epoch_*.pt'))
    command = [sys.executable, 'scripts/train.py', '--config', config, '--run-dir', str(run_dir)]
    if checkpoints: command += ['--resume', str(checkpoints[-1])]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)
for config, run_dir in CONFIGS: train_or_resume(config, run_dir)
os.sync()

In [ ]:
# Evaluate both all-layer models and export their learned 4×12 weight matrices.
for run in (DINO_RUN, VJEPA_RUN):
    metrics = run / 'metrics.json'
    complete = metrics.is_file() and set(json.loads(metrics.read_text())) >= {'camo_test','cod10k_test','chameleon','nc4k'}
    if not complete: subprocess.run([sys.executable, 'scripts/evaluate.py', '--run', str(run)], cwd=PROJECT_DIR, check=True)
    subprocess.run([sys.executable, 'scripts/export_layer_mixture.py', '--run', str(run)], cwd=PROJECT_DIR, check=True)
subprocess.run([sys.executable, 'scripts/compare_runs.py', str(DINO_RUN), str(VJEPA_RUN),
                '--output', str(MIXTURE_COMPARISON), '--qualitative-count', '24'],
               cwd=PROJECT_DIR, check=True)
os.sync()

In [ ]:
# Compare notebook-03 fixed layers with all-12 learned mixtures.
import pandas as pd
from IPython.display import display
from PIL import Image
BASELINE_RUNS = {
    'dinov3_vitb16': RUNS_ROOT / 'phase1_seed42_dinov3_vitb16',
    'vjepa21_vitb16': RUNS_ROOT / 'phase1_seed42_vjepa21_vitb16',
}
MIXTURE_RUNS = {'dinov3_vitb16': DINO_RUN, 'vjepa21_vitb16': VJEPA_RUN}
rows = []
for backbone in BASELINE_RUNS:
    baseline = json.loads((BASELINE_RUNS[backbone] / 'metrics.json').read_text())
    mixture = json.loads((MIXTURE_RUNS[backbone] / 'metrics.json').read_text())
    for dataset in ('camo_test','cod10k_test','chameleon','nc4k'):
        for architecture, values in (('fixed_2-5-8-11', baseline[dataset]), ('all12_mix4', mixture[dataset])):
            rows.append({'backbone': backbone, 'architecture': architecture, 'dataset': dataset,
                         **{key: values[key] for key in ('s_measure','e_adaptive','weighted_f','mae')}})
ablation = pd.DataFrame(rows)
ablation.to_csv(MIXTURE_COMPARISON / 'fixed_vs_all_layer_metrics.csv', index=False)
display(ablation)
display(pd.read_csv(MIXTURE_COMPARISON / 'comparison_metrics.csv'))
display(Image.open(DINO_RUN / 'layer_mixture/layer_mixture_heatmap.png'))
display(Image.open(VJEPA_RUN / 'layer_mixture/layer_mixture_heatmap.png'))
display(Image.open(MIXTURE_COMPARISON / 'metric_comparison.png'))
print('All artifacts:', MIXTURE_COMPARISON)